# 02 — Score de vulnérabilité

Calcul du score de vulnérabilité sanitaire (0-10) pour chaque commune, basé sur 4 composantes normalisées min-max à poids égaux (25% chacune).

**Composantes :**
1. APL inversé (accessibilité potentielle localisée)
2. Précarité économique (revenu médian inversé — proxy de pauvreté, couverture 89% vs 12% pour taux_pauvrete)
3. % population 75+
4. Temps d'accès aux urgences

**Méthode :** Min-max normalisation → moyenne × 10 → classification A-E par quintiles

In [1]:
import os as _os
import pandas as pd
import numpy as np
import json

# Path detection (same pattern as 01_merge)
_cwd = _os.getcwd()
_here = _os.path.dirname(_os.path.abspath("__file__"))

if _os.path.isdir(_os.path.join(_cwd, "notebooks", "data", "processed")):
    PROCESSED = _os.path.join(_cwd, "notebooks", "data", "processed")
elif _os.path.isdir(_os.path.join(_cwd, "data", "processed")):
    PROCESSED = _os.path.join(_cwd, "data", "processed")
else:
    PROCESSED = _os.path.join(_here, "..", "data", "processed")

print(f"PROCESSED path: {PROCESSED}")
_os.makedirs(PROCESSED, exist_ok=True)

df = pd.read_parquet(f"{PROCESSED}/communes_master.parquet")
print(f"Shape: {df.shape}")
print(f"\nNull rates for score components:")
for col in ['apl', 'revenu_median', 'pct_75_plus', 'temps_urgences_min']:
    null_pct = df[col].isna().mean() * 100
    print(f"  {col}: {null_pct:.1f}% null ({df[col].isna().sum()} communes)")
print(f"\n  (ancien: taux_pauvrete: {df['taux_pauvrete'].isna().mean()*100:.1f}% null — remplacé par revenu_median)")


PROCESSED path: /Users/zakariyahamdouchi/Dev/Hackathon/notebooks/data/processed


Shape: (34969, 46)

Null rates for score components:
  apl: 0.4% null (128 communes)
  revenu_median: 10.8% null (3769 communes)
  pct_75_plus: 0.4% null (128 communes)
  temps_urgences_min: 0.4% null (134 communes)

  (ancien: taux_pauvrete: 87.8% null — remplacé par revenu_median)


## 2. Inversion de l'APL

Inversion par `(max - apl)` : évite la division par zéro pour les 517 communes avec APL=0.
Un APL élevé signifie bonne accessibilité → faible vulnérabilité. L'inversion garantit que APL élevé → score bas.

In [2]:
# APL inversion: high APL = low vulnerability, so invert
apl_inv = df['apl'].max() - df['apl']
print(f"APL max: {df['apl'].max():.2f}")
print(f"APL=0 communes (will get max vulnerability): {(df['apl'] == 0).sum()}")
print(f"APL null (will be imputed with median): {df['apl'].isna().sum()}")
print(f"apl_inv range: [{apl_inv.min():.2f}, {apl_inv.max():.2f}]")


APL max: 40.03
APL=0 communes (will get max vulnerability): 517
APL null (will be imputed with median): 128
apl_inv range: [0.00, 40.03]


## 3. Normalisation Min-Max

Chaque composante est normalisée sur [0, 1]. Les valeurs manquantes sont imputées par la **médiane nationale** avant normalisation (impact neutre).

> **Changement v1.1 :** La composante précarité utilise `revenu_median` inversé au lieu de `taux_pauvrete`.
> - `taux_pauvrete` (FiLoSoFi) : seulement 12.2% de couverture (secret statistique petites communes)
> - `revenu_median` (même source FiLoSoFi) : **89.2%** de couverture (seuil de secret plus bas)
> - Corrélation entre les deux : r = -0.74 (proxy solide)
> - Inversion : `(max_revenu - revenu)` → revenu bas = vulnérabilité haute

In [3]:
def minmax_normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# Impute with median BEFORE normalizing (neutral position ~0.5)
score_apl_norm = minmax_normalize(apl_inv.fillna(apl_inv.median()))

# Précarité : revenu_median inversé (bas revenu = haute vulnérabilité)
revenu_inv = df['revenu_median'].max() - df['revenu_median']
score_precarite_norm = minmax_normalize(revenu_inv.fillna(revenu_inv.median()))

score_pct75_norm = minmax_normalize(df['pct_75_plus'].fillna(df['pct_75_plus'].median()))
score_urgences_norm = minmax_normalize(df['temps_urgences_min'].fillna(df['temps_urgences_min'].median()))

print("Normalized component stats:")
for name, s in [('apl_inv', score_apl_norm), ('precarite (revenu inv)', score_precarite_norm),
                ('pct_75', score_pct75_norm), ('urgences', score_urgences_norm)]:
    print(f"  {name}: mean={s.mean():.3f}, std={s.std():.3f}, min={s.min():.3f}, max={s.max():.3f}")

print(f"\nCouverture réelle (non-imputée):")
print(f"  revenu_median: {df['revenu_median'].notna().mean():.1%} (vs taux_pauvrete: {df['taux_pauvrete'].notna().mean():.1%})")


Normalized component stats:
  apl_inv: mean=0.926, std=0.033, min=0.000, max=1.000
  precarite (revenu inv): mean=0.761, std=0.069, min=0.000, max=1.000
  pct_75: mean=0.195, std=0.090, min=0.000, max=1.000
  urgences: mean=0.143, std=0.072, min=0.000, max=1.000

Couverture réelle (non-imputée):
  revenu_median: 89.2% (vs taux_pauvrete: 12.2%)


## 4. Calcul du score

Score = moyenne des 4 composantes normalisées × 10, arrondi à 1 décimale.

**Règle D-01 :** toujours diviser par 4 (poids égaux 25%).
**Règle D-03 :** `score = null` si la commune a moins de 3 des 4 composantes disponibles (avant imputation).

In [4]:
# Count available (non-null) components per commune BEFORE imputation
# revenu_median remplace taux_pauvrete comme composante précarité
n_components = df[['apl', 'revenu_median', 'pct_75_plus', 'temps_urgences_min']].notna().sum(axis=1)

# Raw score: always divide by 4 (equal 25% weights per D-01)
score_raw = (score_apl_norm + score_precarite_norm + score_pct75_norm + score_urgences_norm) / 4

# Scale to 0-10
score_final = (score_raw * 10).round(1)

# Null out communes with <3 components (D-03)
score_final[n_components < 3] = np.nan

# Add to dataframe
df['score'] = score_final
df['n_score_components'] = n_components
df['score_apl_norm'] = score_apl_norm
df['score_pauvrete_norm'] = score_precarite_norm  # garde le nom de colonne pour compatibilité export
df['score_pct75_norm'] = score_pct75_norm
df['score_urgences_norm'] = score_urgences_norm

print(f"Scored communes: {df['score'].notna().sum()} / {len(df)}")
print(f"Null score (<3 components): {df['score'].isna().sum()}")
print(f"Communes with 4 components: {(n_components == 4).sum()} ({(n_components == 4).mean():.1%})")
print(f"\nScore distribution:")
print(df['score'].describe())


Scored communes: 34825 / 34969
Null score (<3 components): 144
Communes with 4 components: 31186 (89.2%)

Score distribution:
count    34825.000000
mean         5.062774
std          0.437006
min          2.500000
25%          4.800000
50%          5.000000
75%          5.300000
max          7.700000
Name: score, dtype: float64


## 5. Classification A-E

Classification inspirée du DPE (Diagnostic de Performance Énergétique).

**Seuils quintiles :** chaque classe regroupe ~20% des communes scorées.
- **A** = moins vulnérable (P0-P20)
- **B** = vulnérabilité faible (P20-P40)
- **C** = vulnérabilité moyenne (P40-P60)
- **D** = vulnérabilité élevée (P60-P80)
- **E** = très vulnérable (P80-P100)

In [5]:
# Quintile thresholds from scored communes only
scored = df['score'].dropna()
thresholds = np.nanpercentile(scored, [20, 40, 60, 80])
print(f"A-E thresholds (quintiles): {thresholds}")

def classify(score, t):
    if pd.isna(score):
        return None
    if score <= t[0]: return 'A'
    if score <= t[1]: return 'B'
    if score <= t[2]: return 'C'
    if score <= t[3]: return 'D'
    return 'E'

df['classe'] = df['score'].apply(lambda s: classify(s, thresholds))

print(f"\nDistribution A-E:")
print(df['classe'].value_counts().sort_index())
print(f"\nNull (no score): {df['classe'].isna().sum()}")


A-E thresholds (quintiles): [4.7 4.9 5.1 5.4]

Distribution A-E:
classe
A    8210
B    6865
C    6611
D    7028
E    6111
Name: count, dtype: int64

Null (no score): 144


## 6. Médianes nationales

Calcul des médianes sur les valeurs brutes (non imputées) pour comparaison par commune dans le JSON final.

> Médiane = plus robuste que la moyenne face aux outliers (D-04).

In [6]:
# National medians from raw (non-imputed) values
median_apl = float(df['apl'].median())
median_revenu = float(df['revenu_median'].median())
median_pct75 = float(df['pct_75_plus'].median())
median_urgences = float(df['temps_urgences_min'].median())

NATIONAL_MEDIANS = {
    'apl': round(median_apl, 2),
    'revenu_median': round(median_revenu, 0),
    'pct_75_seuls': round(median_pct75, 4),
    'temps_urgences_min': round(median_urgences, 1)
}

# Garder aussi l'ancienne clé pauvrete pour compatibilité export
if df['taux_pauvrete'].notna().any():
    NATIONAL_MEDIANS['pauvrete'] = round(float(df['taux_pauvrete'].median()), 4)

print("National medians (raw values):")
for k, v in NATIONAL_MEDIANS.items():
    print(f"  {k}: {v}")

# Save to JSON
medians_path = f"{PROCESSED}/national_medians.json"
with open(medians_path, 'w', encoding='utf-8') as f:
    json.dump(NATIONAL_MEDIANS, f, ensure_ascii=False, indent=2)
print(f"\nSaved to {medians_path}")


National medians (raw values):
  apl: 2.88
  revenu_median: 21450.0
  pct_75_seuls: 0.0922
  temps_urgences_min: 22.0
  pauvrete: 0.12

Saved to /Users/zakariyahamdouchi/Dev/Hackathon/notebooks/data/processed/national_medians.json


## 7. Validation

In [7]:
# Assertions
assert df['score'].notna().sum() > 34000, f"Score computed for {df['score'].notna().sum()} communes (expected >34000)"
assert df['score'].dropna().between(0, 10).all(), "Score out of [0, 10] range"
assert (df['score'][n_components < 3].isna()).all(), "Communes with <3 components should have null score"
assert df['classe'].dropna().isin(['A', 'B', 'C', 'D', 'E']).all(), "Invalid classification"
assert set(df['classe'].dropna().unique()) == {'A', 'B', 'C', 'D', 'E'}, "Not all classes represented"

# Verify medians file
import pathlib
assert pathlib.Path(f"{PROCESSED}/national_medians.json").exists(), "national_medians.json missing"
medians = json.load(open(f"{PROCESSED}/national_medians.json"))
assert 'revenu_median' in medians, "revenu_median missing from national medians"

# Verify score spread improved vs old version
score_std = df['score'].std()
score_range = df['score'].max() - df['score'].min()
print(f"Score std: {score_std:.3f} (ancien: 0.361)")
print(f"Score range: {score_range:.1f} (ancien: 5.1)")

print("\nALL ASSERTIONS PASSED")
print(f"\nFinal stats:")
print(f"  Scored: {df['score'].notna().sum()} communes")
print(f"  Score range: [{df['score'].min():.1f}, {df['score'].max():.1f}]")
print(f"  Score std: {score_std:.3f}")
print(f"  Classes: {dict(df['classe'].value_counts().sort_index())}")


Score std: 0.437 (ancien: 0.361)
Score range: 5.2 (ancien: 5.1)

ALL ASSERTIONS PASSED

Final stats:
  Scored: 34825 communes
  Score range: [2.5, 7.7]
  Score std: 0.437
  Classes: {'A': np.int64(8210), 'B': np.int64(6865), 'C': np.int64(6611), 'D': np.int64(7028), 'E': np.int64(6111)}


## 8. Sauvegarde

In [8]:
df.to_parquet(f"{PROCESSED}/communes_master.parquet", index=False)
print(f"Saved: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nNew columns added: score, classe, n_score_components, score_apl_norm, score_pauvrete_norm, score_pct75_norm, score_urgences_norm")
print(f"\nAll columns: {list(df.columns)}")


Saved: 34969 rows, 46 columns

New columns added: score, classe, n_score_components, score_apl_norm, score_pauvrete_norm, score_pct75_norm, score_urgences_norm

All columns: ['code_commune', 'nom_commune', 'population', 'code_departement', 'nb_generalistes', 'nb_medecins_total', 'nb_specialistes', 'specialistes_detail', 'apl_2022', 'apl_2023', 'apl', 'apl_evolution', 'taux_pauvrete', 'revenu_median', 'pct_75_plus', 'temps_urgences_min', 'temps_urgences_mcs_min', 'nb_etablissements', 'has_hopital', 'has_ehpad', 'nb_msp', 'has_msp', 'latitude', 'longitude', 'prev_diabete', 'prev_cardio', 'prev_psy', 'prev_cancers', 'prev_respiratoire', 'score', 'n_score_components', 'score_apl_norm', 'score_pauvrete_norm', 'score_pct75_norm', 'score_urgences_norm', 'classe', 'region', 'surface_ha', 'densite_hab_km2', 'manques', 'pct_55plus_dept', 'medecins_55plus_est', 'domino_alert', 'projection_2030', 'log_pop', 'jumelles']
